# MV AutoML Studio dentro de un notebook de Fabric

Este notebook entrena un modelo sobre una tabla del Lakehouse y deja los
resultados en archivos que Power BI lee directo. No levanta ningún servidor: usa
el mismo motor del programa de escritorio, con el mismo protocolo de validación
(entrenamiento · selección · **holdout ciego**).

**Antes de correrlo**, copiá la carpeta `backend` del programa a
`Files/mv-automl/backend` del Lakehouse (se sube desde el panel de Fabric, o con
`notebookutils.fs.cp`). Es lo único que hace falta instalar.

El notebook también corre tal cual en tu máquina: si no encuentra el Lakehouse,
arma una tabla de ejemplo y escribe en una carpeta local. Así se prueba el
circuito completo antes de tocar un dato de la empresa.


In [ ]:
# ── 1. Dejar el programa al alcance del notebook ─────────────────────────────
import sys
from pathlib import Path

RUTA_PROGRAMA = Path("/lakehouse/default/Files/mv-automl/backend")
if RUTA_PROGRAMA.exists() and str(RUTA_PROGRAMA) not in sys.path:
    sys.path.append(str(RUTA_PROGRAMA))

from app import notebook as mv        # noqa: E402

# Dónde escribir: el Lakehouse montado si estamos en Fabric, una carpeta local si no.
LAKEHOUSE = Path("/lakehouse/default/Files")
SALIDA = (LAKEHOUSE / "mv" if LAKEHOUSE.exists() else Path("salida_mv"))
print("Los resultados van a:", SALIDA)

In [ ]:
# ── 2. Traer la tabla ────────────────────────────────────────────────────────
# En Fabric, del Lakehouse. Fuera de Fabric, una tabla de ejemplo con la misma
# forma: nunca se pone un dato real de la empresa en un archivo de ejemplo.
import numpy as np
import pandas as pd

try:
    datos = spark.sql("SELECT * FROM ventas_clientes").toPandas()   # noqa: F821
    print("Tabla del Lakehouse:", datos.shape)
except Exception as exc:
    print("Sin Lakehouse a mano, uso la tabla de ejemplo.", type(exc).__name__)
    rng = np.random.default_rng(3)
    n = 800
    zona = rng.choice(["norte", "sur", "centro"], n)
    visitas = rng.integers(1, 12, n)
    prob = np.clip(0.12 + 0.05 * visitas + np.where(zona == "centro", 0.12, 0.0), 0, 0.95)
    datos = pd.DataFrame({
        "cliente_id": [f"c{i:05d}" for i in range(n)],
        "zona": zona,
        "visitas": visitas,
        "descuento": rng.uniform(0, 0.3, n).round(3),
        "antiguedad_meses": rng.integers(1, 60, n),
        "compro": rng.binomial(1, prob, n),
    })

datos.head()

In [ ]:
# ── 3. Entrenar ──────────────────────────────────────────────────────────────
# `excluir` saca los identificadores: no predicen nada y ensucian la explicación.
# Si la tabla tiene fecha, pasá `tiempo="fecha"` y las tres ventanas quedan
# consecutivas en el tiempo, que es como se valida un modelo que va a producción.
modelo = mv.entrenar(
    datos,
    objetivo="compro",
    excluir=["cliente_id"],
    presupuesto_segundos=20,
    progreso=True,
)

resumen = modelo.resumen()
print(resumen["veredicto"])
resumen

In [ ]:
# ── 4. Leer el resultado ─────────────────────────────────────────────────────
# El leaderboard muestra qué midió cada familia en la ventana de selección y en
# el holdout ciego. La brecha entre las dos es lo que dice si el modelo se
# sostiene fuera de la ventana con la que se lo eligió.
print(modelo.tabla_modelos().to_string(index=False))
print()
print(modelo.importancias(n=8).to_string(index=False))

In [ ]:
# ── 5. Guardar el modelo y dejar la salida para Power BI ─────────────────────
carpeta_modelo = modelo.guardar(SALIDA / "modelo_compras")
archivos = modelo.para_powerbi(SALIDA / "powerbi", datos=datos, conservar=["cliente_id", "zona"])

print("Modelo guardado en:", carpeta_modelo)
for nombre, ruta in archivos.items():
    print(f"  {nombre:14s} → {ruta}")

## Cómo lo toma Power BI

En el Lakehouse quedan cuatro tablas en `Files/mv/powerbi`:

| Archivo | Una fila por | Para qué sirve |
|---|---|---|
| `predicciones.parquet` | caso | la lista priorizada: probabilidad por cliente |
| `metricas.parquet` | métrica y ventana | comparar selección contra holdout en un visual |
| `importancias.parquet` | variable | qué sostiene al modelo, medido sobre el holdout |
| `resumen.parquet` | modelo | la tarjeta del tablero: métrica, brecha y veredicto |

En Power BI Desktop: **Obtener datos → Lakehouse (OneLake)**, elegís la carpeta
`Files/mv/powerbi` y cargás los cuatro archivos. Con *Direct Lake* el informe se
actualiza solo la próxima vez que el notebook vuelva a escribir.

Las métricas vienen en formato largo (`metrica · ventana · valor`) justamente
para que un visual salga sin pivotear nada a mano.

## Volver a usar el modelo mañana

No hace falta reentrenar para puntuar datos nuevos:

```python
modelo = mv.cargar("/lakehouse/default/Files/mv/modelo_compras")
nuevos = spark.sql("SELECT * FROM ventas_clientes_hoy").toPandas()
modelo.para_powerbi("/lakehouse/default/Files/mv/powerbi", datos=nuevos,
                    conservar=["cliente_id"])
```

Ese bloque es el que conviene programar en una *pipeline* de Fabric: el modelo
se entrena cuando se decide, y el scoring corre todos los días.


In [ ]:
# ── 6. Aplicar el modelo a filas nuevas ──────────────────────────────────────
# Se vuelve a cargar desde el disco a propósito: es el camino que va a correr la
# pipeline mañana, sin el objeto que quedó en memoria de este notebook.
recargado = mv.cargar(carpeta_modelo)
prediccion = recargado.predecir(datos.head(10), conservar=["cliente_id"])
print(prediccion.to_string(index=False))